# LiteLLM SDK Proof of Concept

Testing basic text responses and structured outputs with GPT-4o-mini and Claude Haiku.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'))

assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set'
assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY not set'
print('Keys loaded.')

Keys loaded.


## 1. Basic Text Response

In [2]:
from litellm import acompletion


def print_response(response):
    print(response.choices[0].message.content)
    cost = response._hidden_params.get('response_cost', 0)
    print(f"Cost: ${cost:.6f}")
    print(f"Tokens: in={response.usage.prompt_tokens} out={response.usage.completion_tokens} total={response.usage.total_tokens}")


messages = [{"role": "user", "content": "In one sentence, what is a bookmark manager?"}]

# GPT-4o-mini
response = await acompletion(model="gpt-4o-mini", messages=messages)
print("=== GPT-4o-mini ===")
print_response(response)

=== GPT-4o-mini ===
A bookmark manager is a tool or application that helps users organize, save, and access their favorite web links or bookmarks in a structured manner.
Cost: $0.000019
Tokens: in=17 out=28 total=45


In [3]:
# Claude Haiku
response = await acompletion(model="anthropic/claude-haiku-4-5-20251001", messages=messages)
print("=== Claude Haiku 4.5 ===")
print_response(response)

=== Claude Haiku 4.5 ===
A bookmark manager is a tool that helps you organize, store, and quickly access your saved website links.
Cost: $0.000137
Tokens: in=17 out=24 total=41


In [4]:
# Gemini Flash Lite
response = await acompletion(model="gemini/gemini-2.5-flash-lite", messages=messages)
print("=== Gemini 2.5 Flash Lite ===")
print_response(response)

=== Gemini 2.5 Flash Lite ===
A bookmark manager is a tool that helps you organize, save, and easily access your favorite websites and web pages.
Cost: $0.000010
Tokens: in=11 out=23 total=34


## 2. Structured Output with Pydantic

In [5]:
from pydantic import BaseModel


class TagSuggestions(BaseModel):
    tags: list[str]
    reasoning: str


prompt = (
    "Suggest tags for a bookmark with the following details:\n"
    "Title: Introduction to FastAPI\n"
    "URL: https://fastapi.tiangolo.com/tutorial/\n"
    "Description: FastAPI framework, high performance, easy to learn."
)
messages = [{"role": "user", "content": prompt}]

In [6]:
# GPT-4o-mini with Pydantic structured output
response = await acompletion(
    model="gpt-4o-mini",
    messages=messages,
    response_format=TagSuggestions,
)
print("=== GPT-4o-mini (structured) ===")
parsed = TagSuggestions.model_validate_json(response.choices[0].message.content)
print(f"Tags: {parsed.tags}")
print(f"Reasoning: {parsed.reasoning}")
print_response(response)

=== GPT-4o-mini (structured) ===
Tags: ['FastAPI', 'Web Development', 'Python', 'Framework', 'API', 'High Performance', 'Learning', 'Programming', 'Tutorial']
Reasoning: These tags reflect the content and purpose of the bookmark. 'FastAPI' and 'Framework' are directly related to the title and subject. 'Web Development' and 'API' describe the application domain. 'Python' is the programming language used. 'High Performance' highlights a key feature of FastAPI, while 'Learning' and 'Tutorial' signify that it's a resource for gaining knowledge.
{"tags":["FastAPI","Web Development","Python","Framework","API","High Performance","Learning","Programming","Tutorial"],"reasoning":"These tags reflect the content and purpose of the bookmark. 'FastAPI' and 'Framework' are directly related to the title and subject. 'Web Development' and 'API' describe the application domain. 'Python' is the programming language used. 'High Performance' highlights a key feature of FastAPI, while 'Learning' and 'Tutor

In [7]:
# Gemini Flash Lite with Pydantic structured output
response = await acompletion(
    model="gemini/gemini-2.5-flash-lite",
    messages=messages,
    response_format=TagSuggestions,
)
print("=== Gemini 2.5 Flash Lite (structured) ===")
parsed = TagSuggestions.model_validate_json(response.choices[0].message.content)
print(f"Tags: {parsed.tags}")
print(f"Reasoning: {parsed.reasoning}")
print_response(response)

=== Gemini 2.5 Flash Lite (structured) ===
Tags: ['python', 'web framework', 'API', 'FastAPI', 'tutorial', 'performance']
Reasoning: The title 'Introduction to FastAPI' clearly indicates the main subject. The URL points to the official FastAPI tutorial, so 'tutorial' is relevant. The description highlights 'FastAPI framework', 'high performance', and 'easy to learn', which translate into tags like 'web framework', 'API', 'performance', and the framework name itself. Since FastAPI is a Python framework, 'python' is a core tag.
{
  "tags": ["python", "web framework", "API", "FastAPI", "tutorial", "performance"],
  "reasoning": "The title 'Introduction to FastAPI' clearly indicates the main subject. The URL points to the official FastAPI tutorial, so 'tutorial' is relevant. The description highlights 'FastAPI framework', 'high performance', and 'easy to learn', which translate into tags like 'web framework', 'API', 'performance', and the framework name itself. Since FastAPI is a Python fr

In [8]:
# Claude Haiku with Pydantic structured output
response = await acompletion(
    model="anthropic/claude-haiku-4-5-20251001",
    messages=messages,
    response_format=TagSuggestions,
)
print("=== Claude Haiku 4.5 (structured) ===")
parsed = TagSuggestions.model_validate_json(response.choices[0].message.content)
print(f"Tags: {parsed.tags}")
print(f"Reasoning: {parsed.reasoning}")
print_response(response)

=== Claude Haiku 4.5 (structured) ===
Tags: ['FastAPI', 'tutorial', 'web-framework', 'Python', 'REST-API', 'learning', 'documentation']
Reasoning: The bookmark is about FastAPI, which is a modern Python web framework for building APIs. Based on the title "Introduction to FastAPI", description mentioning "framework, high performance, easy to learn", and the URL pointing to the official tutorial documentation, I've suggested tags that cover: the specific technology (FastAPI), the content type (tutorial, documentation), the broader category (web-framework, REST-API), the programming language (Python), and the purpose (learning). These tags would help with organizing and searching for this bookmark in the future.
{"tags": ["FastAPI", "tutorial", "web-framework", "Python", "REST-API", "learning", "documentation"], "reasoning": "The bookmark is about FastAPI, which is a modern Python web framework for building APIs. Based on the title \"Introduction to FastAPI\", description mentioning \"fra

## 3. Streaming — Cost Verification

Check that cost/token data is available after consuming a streaming response.

In [12]:
from litellm import acompletion

messages = [{"role": "user", "content": "In one sentence, what is a bookmark manager?"}]

for model_name in ["gpt-4o-mini", "anthropic/claude-haiku-4-5-20251001", "gemini/gemini-2.5-flash-lite"]:
    print(f"=== {model_name} (streaming) ===")
    response = await acompletion(model=model_name, messages=messages, stream=True)

    full_text = ""
    async for chunk in response:
        delta = chunk.choices[0].delta.content
        if delta:
            full_text += delta

    # After stream is consumed, check for usage/cost on the final chunk
    print(f"Response: {full_text[:80]}...")
    usage = getattr(chunk, 'usage', None)
    print(f"Final chunk usage: {usage}")
    cost = chunk._hidden_params.get('response_cost', None) if hasattr(chunk, '_hidden_params') else None
    print(f"Final chunk cost: {cost}")

    # Also check the stream_tracker if available
    tracker = getattr(response, 'completion_stream', None)
    if hasattr(response, '_hidden_params'):
        print(f"Stream _hidden_params keys: {list(response._hidden_params.keys())}")
        stream_cost = response._hidden_params.get('response_cost', None)
        print(f"Stream response_cost: {stream_cost}")
    print()

=== gpt-4o-mini (streaming) ===
Response: A bookmark manager is a tool or application that allows users to organize, store...
Final chunk usage: Usage(completion_tokens=31, prompt_tokens=17, total_tokens=48, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None))
Final chunk cost: None
Stream _hidden_params keys: ['model_id', 'api_base', 'additional_headers', 'litellm_call_id', 'response_cost', 'litellm_model_name', '_response_ms', 'litellm_overhead_time_ms', 'callback_duration_ms']
Stream response_cost: 0.0

=== anthropic/claude-haiku-4-5-20251001 (streaming) ===
Response: A bookmark manager is a tool that helps you organize, store, and quickly access ...
Final chunk usage: None
Final chunk cost: 

In [13]:
from litellm import acompletion, completion_cost

messages = [{"role": "user", "content": "In one sentence, what is a bookmark manager?"}]

for model_name in ["gpt-4o-mini", "anthropic/claude-haiku-4-5-20251001", "gemini/gemini-2.5-flash-lite"]:
    print(f"=== {model_name} (streaming + manual cost) ===")
    response = await acompletion(model=model_name, messages=messages, stream=True)

    chunks = []
    full_text = ""
    async for chunk in response:
        chunks.append(chunk)
        delta = chunk.choices[0].delta.content
        if delta:
            full_text += delta

    # Try completion_cost with the final chunk (has usage for OpenAI)
    final_chunk = chunks[-1]
    try:
        cost = completion_cost(model=model_name, completion_response=final_chunk)
        print(f"completion_cost(final_chunk): ${cost:.6f}")
    except Exception as e:
        print(f"completion_cost(final_chunk) failed: {e}")

    # Try with prompt/completion text lengths as fallback
    try:
        cost = completion_cost(
            model=model_name,
            prompt="In one sentence, what is a bookmark manager?",
            completion=full_text,
        )
        print(f"completion_cost(text): ${cost:.6f}")
    except Exception as e:
        print(f"completion_cost(text) failed: {e}")
    print()

=== gpt-4o-mini (streaming + manual cost) ===
completion_cost(final_chunk): $0.000016
completion_cost(text): $0.000015

=== anthropic/claude-haiku-4-5-20251001 (streaming + manual cost) ===
completion_cost(final_chunk): $0.000000
completion_cost(text): $0.000125

=== gemini/gemini-2.5-flash-lite (streaming + manual cost) ===
completion_cost(final_chunk): $0.000000
completion_cost(text): $0.000011

